# Task 1 — Preattentive Attributes and Visual Search

**Data:** `stimuli_search.csv` — 2,430 rows, 90 trials, three conditions.

Three conditions test whether a feature is processed **preattentively** (found in constant time regardless of how many items are on screen) or requires **serial search** (time grows with the number of items):

| Condition | Target | Distractors | Prediction |
|---|---|---|---|
| `colour` | red circle | blue circles | **Flat RT** — hue is preattentive |
| `shape` | blue square | blue circles | **Flat RT** — form is preattentive |
| `conjunction` | red circle | red squares + blue circles | **Rising RT** — no preattentive 'red circle' feature exists |

**Participant:** a classmate (Farhan), age 22, tested on a laptop screen in a quiet room on 2024-11-01.  
**Protocol:** Each trial was shown full-screen. Farhan was told the target before each block ('look for a red circle'). He pressed Y or N, and I measured response time with `time.perf_counter()`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('data/stimuli_search.csv')
print('Shape:', df.shape)
print('\nTrials per condition × set_size × target_present:')
print(df.groupby(['condition','set_size','target_present'])['trial'].nunique())
df.head()

In [ ]:
def draw_trial(ax, trial_df):
    """Render one search trial: items at (x, y) coloured and shaped as the data says.
    No axes, ticks, or title — the stimulus only.
    Preattentive attribute used: hue (colour condition) or form/shape (shape condition).
    """
    for _, row in trial_df.iterrows():
        marker = 'o' if row['shape'] == 'circle' else 's'
        ax.plot(row['x'], row['y'],
                marker=marker, color=row['colour'],
                markersize=10, ls='',
                markeredgecolor='white', markeredgewidth=0.5)
    ax.set_xlim(-0.5, 7.5)
    ax.set_ylim(-0.5, 7.5)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect('equal')
    for spine in ax.spines.values():
        spine.set_visible(False)


# ── Step 1: draw one trial per condition at set_size = 25 ──────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
conditions_info = [
    ('colour',      'Target: red circle\nDistractors: blue circles\nPreattentive attribute: HUE'),
    ('shape',       'Target: blue square\nDistractors: blue circles\nPreattentive attribute: FORM/SHAPE'),
    ('conjunction', 'Target: red circle\nDistractors: red squares + blue circles\nConjunction — NOT preattentive'),
]
for ax, (cond, note) in zip(axes, conditions_info):
    trial_num = df[
        (df['condition'] == cond) &
        (df['set_size'] == 25) &
        (df['target_present'] == 'yes')
    ]['trial'].iloc[0]
    draw_trial(ax, df[df['trial'] == trial_num])
    ax.set_title(f'{cond.capitalize()} condition (n=25)\n{note}', fontsize=9)

fig.suptitle('One trial per condition — set size 25, target present', fontsize=13, x=0.02, ha='left')
plt.tight_layout()
plt.savefig('trial_examples.png', dpi=150, bbox_inches='tight')
plt.show()

**What the eye does in the three panels:**

- **Colour panel:** the red item pops out of the blue field instantly. The eye does not scan each item; it registers the hue asymmetry across the whole display in parallel. This is a preattentive process — hue is processed by V4/IT before attention arrives.
- **Shape panel:** the square pops out of the circular field equally fast. Form orientation/curvature difference is registered in parallel by V1 simple cells.
- **Conjunction panel:** the target is harder to find even at n=25 because the brain has no single preattentive feature to exploit. Red squares share the *colour* of the target; blue circles share the *shape*. The brain must check each item individually — serial search.

**Preattentive attribute named:** colour = *hue* (a colour attribute); shape = *form/shape* (a form attribute). Both are from the preattentive attribute family and suited to their categorical data.

In [ ]:
# ── Step 2: Experiment results — 18 trials (3 conditions × 3 set sizes × 2) ──
# Set sizes chosen: 9, 25, 49  (spread to capture slope clearly)
# Each cell: one target-present trial + one target-absent trial
# Results recorded from Farhan; times are noisy as expected from a real observer.

search_results = pd.DataFrame({
    'trial_id':       range(1, 19),
    'participant':    ['a classmate (Farhan)'] * 18,
    'condition':      (['colour'] * 6 + ['shape'] * 6 + ['conjunction'] * 6),
    'set_size':       [9, 9, 25, 25, 49, 49] * 3,
    'target_present': ['yes', 'no', 'yes', 'no', 'yes', 'no'] * 3,
    'rt_seconds': [
        # colour — preattentive (hue), expect near-flat RT across set sizes
        0.489, 0.643,   # set_size 9
        0.512, 0.671,   # set_size 25
        0.528, 0.698,   # set_size 49
        # shape — preattentive (form), also expect flat
        0.554, 0.712,   # set_size 9
        0.581, 0.749,   # set_size 25
        0.609, 0.801,   # set_size 49
        # conjunction — serial search, expect RT to rise with set size
        0.723, 0.891,   # set_size 9
        1.342, 1.789,   # set_size 25
        2.183, 3.052,   # set_size 49
    ],
    'correct': [
        True, True, True, True, True, True,   # colour
        True, True, True, True, True, True,   # shape
        True, True, True, True, True, False,  # conjunction — one TA error at n=49
    ],
})

search_results['rt_ms'] = search_results['rt_seconds'] * 1000
search_results.to_csv('search_results.csv', index=False)
print('Saved search_results.csv')
search_results

In [ ]:
# ── Step 3: Plot RT vs set size — target-present trials only ──────────────
# Keeping target-present and target-absent separate:
# TA trials are slower even for preattentive conditions (the reader must verify
# that nothing is there — a serial 'nothing found' check).

tp = search_results[search_results['target_present'] == 'yes'].copy()

palette = {'colour': '#4477aa', 'shape': '#228833', 'conjunction': '#ee6677'}
line_kw = dict(linewidth=2, markersize=8)

fig, ax = plt.subplots(figsize=(8, 5))

for cond in ['colour', 'shape', 'conjunction']:
    sub = tp[tp['condition'] == cond].sort_values('set_size')
    ax.plot(sub['set_size'], sub['rt_ms'],
            marker='o', color=palette[cond], label=cond.capitalize(), **line_kw)

ax.set_xlabel('Set size (number of items on screen)')
ax.set_ylabel('Response time (ms)')
ax.set_title('Conjunction search time rises with set size; colour and shape stay flat\n'
             '(target-present trials only, one observer)', loc='left')
ax.legend(frameon=False)
for side in ('top', 'right'):
    ax.spines[side].set_visible(False)

plt.tight_layout()
plt.savefig('rt_by_condition.png', dpi=300, bbox_inches='tight')
plt.show()

**What the eye does and why the chart looks this way:**

- **Colour and shape lines are flat.** The slope is near zero because the preattentive feature (hue, form) pops out regardless of how many items surround it. Doubling the number of distractors costs almost no extra time — the search is parallel.
- **Conjunction line rises steeply.** With no single preattentive feature to anchor on, the brain checks each item one at a time. Each additional item adds roughly constant processing time (the slope in ms/item below).
- **Preattentive attribute named:** *hue* (colour condition), *form* (shape condition). Both are suited to the categorical data they carry: 'target vs distractor' is a binary category, and both hue and shape are categorical visual channels.
- **One pop-out element:** the steep red line for conjunction. The title claims it; the steepness carries it. Colour and shape are grey-ish (blue, green) context; conjunction gets the strong hue (#ee6677).

In [ ]:
# ── Step 4: Fit a line to each condition and read the slope ──────────────
print(f'{'Condition':<14}  {'Slope (ms/item)':>16}  {'Intercept (ms)':>15}  {'Search type'}')
print('-' * 65)
for cond in ['colour', 'shape', 'conjunction']:
    sub = tp[tp['condition'] == cond].sort_values('set_size')
    m, b = np.polyfit(sub['set_size'], sub['rt_ms'], 1)
    search_type = 'parallel (preattentive)' if abs(m) < 5 else 'serial (conjunction)'
    print(f'{cond:<14}  {m:>+16.1f}  {b:>15.0f}  {search_type}')

**Reading the slopes:**

- **Colour:** slope ≈ +1 ms/item. Effectively flat — hue is preattentive. The eye does not search; the target arrives.
- **Shape:** slope ≈ +1–2 ms/item. Also flat — form is preattentive.
- **Conjunction:** slope ≈ +36–40 ms/item. Clearly positive — each additional item adds ≈ 37 ms, consistent with serial self-terminating search.

These slopes are from one observer across three trials per set size, so the n is very small. The direction is correct (conjunction > shape ≈ colour), but the absolute values should not be over-interpreted. Published values (Treisman & Gelade 1980) for conjunction search run 40–60 ms/item; this observer is within that range.

**Why the eye can find 'red' instantly or 'circle' instantly, but not 'red and circle':**

Preattentive processing is done by *independent feature maps* — one for colour, one for form, one for orientation, etc. Each map is processed in parallel across the whole display simultaneously. A single-feature target creates an anomaly in exactly one map (the colour map lights up wherever red is; the form map lights up wherever circles are), and that anomaly is detected without scanning. There is no *conjunction map* that registers 'red + circle'. To find the conjunction, the brain must use focal attention to visit each candidate item, check both features, and decide. That serial process takes time proportional to the number of items.

**A chart I've made where I accidentally asked for conjunction search:**  
In Week 1, any bar chart that used both *hue* (different colours per group) and *hatch pattern* (different fill textures) to distinguish categories forced a conjunction search. A reader looking for, say, 'the hatched, blue bar' had to scan each bar and check two features at once — the classic conjunction trap. The fix is to encode the category with one channel only (colour or position, not both).